# Dataset

In [104]:
import pandas as pd
import numpy as np
import re

def extract_top_features(code):
    if not code or not code.strip():
        return [0.0, 0.0, 0.0, 0.0]

    # Feat 1: blank lines ratio
    lines = code.strip().split('\n')
    total_lines = len(lines)
    
    blank_lines = sum(1 for line in lines if not line.strip())
    blank_line_ratio = blank_lines / total_lines if total_lines > 0 else 0.0

    # Feat 2: whitespace ratio
    whitespaces = [l.count(' ') + 4 * l.count('\t') for l in lines]
    mean_whitespaces = np.mean(whitespaces) if len(whitespaces) > 0 else 0
    whitespace_ratio = np.std(whitespaces) / mean_whitespaces if mean_whitespaces > 0 else 0.0

    # Feat 3: comment ratio
    # Find '//' and '#'
    comment_lines = sum(1 for line in lines if re.search(r'(?://|#)', line))
    comment_ratio = comment_lines / total_lines if total_lines > 0 else 0.0

    # Feat 4: paragraph
    pr_lines = [len(pr.split('\n')) for pr in code.split("\n\n")]
    paragraph_ratio = (np.std(pr_lines)) / (np.mean(pr_lines) + np.std(pr_lines)) if len(pr_lines) > 0 else 0.0

    return [
        round(blank_line_ratio, 4),
        round(whitespace_ratio, 4),
        round(comment_ratio, 4),
        round(paragraph_ratio, 4)
    ]

def get_full_dataset(file_path: str, return_labels=True):
    columns = ["code", "label"] if return_labels else ["code"]
    df = pd.read_parquet(file_path, columns=columns)
    codes = df["code"].tolist()
    feats = np.array([extract_top_features(c) for c in codes])

    if (return_labels is False):
        return feats

    labels = df["label"].to_numpy()
    return feats, labels

def get_full_dataset_from_file(file_path: str):
    data = pd.read_parquet(file_path).to_numpy()
    return data[:, :-1], data[:, -1]
    

In [105]:
# -- Re-extract features
# train_feats, train_labels = get_full_dataset('../data/train.parquet')
# valid_feats, valid_labels = get_full_dataset('../data/validation.parquet')
# test_feats, test_labels = get_full_dataset('../data/test_sample.parquet')

# -- Load extracted features
train_feats, train_labels = get_full_dataset_from_file('../features/extracted_features/train.parquet')
valid_feats, valid_labels = get_full_dataset_from_file('../features/extracted_features/validation.parquet')
test_feats, test_labels = get_full_dataset_from_file('../features/extracted_features/test_sample.parquet')

In [ ]:
# Save extracted features to file
df = pd.DataFrame(np.hstack([train_feats, train_labels.reshape(-1, 1)]), columns=['blank_line','whitespace','comment', 'paragraph_lines', 'label'])
df.to_parquet('../features/extracted_features/train.parquet')

df = pd.DataFrame(np.hstack([valid_feats, valid_labels.reshape(-1, 1)]), columns=['blank_line','whitespace','comment', 'paragraph_lines', 'label'])
df.to_parquet('../features/extracted_features/validation.parquet')

df = pd.DataFrame(np.hstack([test_feats, test_labels.reshape(-1, 1)]), columns=['blank_line','whitespace','comment', 'paragraph_lines', 'label'])
df.to_parquet('../features/extracted_features/test_sample.parquet')

# Model

In [106]:
import torch
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, num_feats=1):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(num_feats, 16),
            nn.GELU(),
            nn.Linear(16, 8),
            nn.GELU(),
            nn.Linear(8, 4),
            nn.GELU(),
            nn.Linear(4, 2)
        )
    def forward(self, X):
        return self.fc(X)

# Training

In [113]:
def train_model(model, train_dataset: torch.tensor, valid_dataset: torch.tensor, num_feats = 3, batch_size=512, epochs=5, lr=1e-4, device="cpu"):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    train_data_size, valid_data_size = len(train_dataset), len(valid_dataset)
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        correct = 0
        
        for i in range(0, train_data_size, batch_size):
            batch = train_dataset[i:i+batch_size]
            X, y = batch[:, :-1], batch[:, -1]
            X, y = X.to(device), y.to(device).long()

            optimizer.zero_grad()
            logits = model(X)

            # Backprop
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * X.size(0)

            # Check predictions
            preds = torch.argmax(logits, dim=1)
            correct += (preds == y).sum().item()

        train_loss /= train_data_size
        train_acc = correct / train_data_size

        # -- Validation
        model.eval()
        val_loss = 0.0
        correct = 0
        
        with torch.no_grad():
            for i in range(0, valid_data_size, batch_size):
                batch = valid_dataset[i:i+batch_size]
                X, y = batch[:, :-1], batch[:, -1]
                X, y = X.to(device), y.to(device).long()

                # Valid loss
                logits = model(X)
                loss = criterion(logits, y)
                val_loss += loss.item() * X.size(0)

                # Valid accuracy
                preds = torch.argmax(logits, dim=1)
                correct += (preds == y).sum().item()
            
            val_loss /= valid_data_size
            val_acc = correct / valid_data_size

            print(f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")



In [126]:
# model = MLP(num_feats=4) # <-- Comment here for resuming training
_train_dataset = torch.tensor(np.hstack([train_feats[:200000, :], train_labels[:200000].reshape(-1, 1)]), dtype=torch.float32)
_valid_dataset = torch.tensor(np.hstack([test_feats[:1000, :], test_labels[:1000].reshape(-1, 1)]), dtype=torch.float32)
train_model(model, _train_dataset, _valid_dataset, num_feats=num_feats, device="cuda", lr=1e-6, epochs=5, batch_size=512)

Epoch 1/5 | Train Loss: 0.4654 Acc: 0.7889 | Val Loss: 0.6737 Acc: 0.7160
Epoch 2/5 | Train Loss: 0.4621 Acc: 0.7922 | Val Loss: 0.6772 Acc: 0.7190
Epoch 3/5 | Train Loss: 0.4590 Acc: 0.7954 | Val Loss: 0.6813 Acc: 0.7180
Epoch 4/5 | Train Loss: 0.4560 Acc: 0.7979 | Val Loss: 0.6859 Acc: 0.7200
Epoch 5/5 | Train Loss: 0.4532 Acc: 0.8002 | Val Loss: 0.6908 Acc: 0.7190


# Run tests

In [110]:
def testing_predict(model, test_dataset: torch.tensor, batch_size=64):
    test_data_size = len(test_dataset)

    model.eval()
    device = "cuda"

    test_predictions = []
    with torch.no_grad():
        for i in range(0, test_data_size, batch_size):
            batch = test_dataset[i:i+batch_size]
            X = batch[:, :].to(device)

            logits = model(X)
            preds = torch.argmax(logits, dim=1)
            test_predictions.extend(preds.cpu().tolist())

    return test_predictions

### Test Sample

In [127]:
test_feats, test_labels = get_full_dataset('../data/test_sample.parquet')
test_feats = torch.tensor(test_feats, dtype=torch.float32)

test_predictions = testing_predict(model, test_feats[:, :])

# Accuracy
correct = np.sum(test_labels == test_predictions)
print("Accuracy: ", correct / len(test_labels))

# Macro F1 Score
from sklearn.metrics import f1_score
def macro_f1(preds, labels):
    return f1_score(labels, preds, average='macro')

print("F1 score: ", macro_f1(test_predictions, test_labels))

Accuracy:  0.719
F1 score:  0.67848050573529


### Real test -> Submission

In [97]:
import pandas as pd
# test_X = get_full_dataset('../data/test.parquet', return_labels=False)
test_X = pd.read_parquet('../features/extracted_features/test.parquet').to_numpy()
test_X = torch.tensor(test_X[:], dtype=torch.float32)

test_predictions = testing_predict(model, test_X)
ids = pd.read_parquet('../data/test.parquet', columns=["ID"])["ID"].values

df = pd.DataFrame({
    "ID": ids,   
    "label": test_predictions
})

df.to_csv("submission.csv", index=False)

### Save test features to test.parquet

In [ ]:
test_X = get_full_dataset('../data/test.parquet', return_labels=False)
df = pd.DataFrame(test_X, columns=['blank_line','whitespace','comment', 'paragraph_lines'])
df.to_parquet('../features/extracted_features/test.parquet')

# Others (Khôi không dùng đến) ---------------

In [53]:
print("Train: ", train_dataset[:, :5].mean(axis=0), train_dataset[:, :5].std(axis=0))
print("Valid: ", validation_dataset[:, :5].mean(axis=0), validation_dataset[:, :5].std(axis=0))
print("Test: ", test_dataset[:, :5].mean(axis=0), test_dataset[:, :5].std(axis=0))

Train:  [0.62468957 0.72006535 0.13615315 0.53160718 0.75123857] [0.26437125 0.21645095 0.11607943 0.17131097 0.46461197]
Valid:  [0.62507329 0.72079514 0.13578454 0.53189979 0.75011993] [0.26683663 0.21746957 0.11569903 0.16934744 0.4658992 ]
Test:  tensor([0.6819, 1.2963, 0.0904, 0.4339, 0.9633]) tensor([0.2028, 0.4085, 0.1037, 0.1435, 0.4653])


In [112]:
print("Train: ", scaled_train[:, :5].mean(axis=0), scaled_train[:, :5].std(axis=0))
print("Valid: ", scaled_valid[:, :5].mean(axis=0), scaled_valid[:, :5].std(axis=0))
print("Test: ", scaled_test[:, :5].mean(axis=0), scaled_test[:, :5].std(axis=0))

Train:  [-4.07290202e-16  1.49106683e-15 -6.96509517e-17 -2.15862883e-16
 -6.04885031e-17] [1. 1. 1. 1. 1.]
Valid:  [ 0.00145145  0.00337162 -0.00317547  0.00170809 -0.00240768] [1.00932544 1.00470601 0.99672294 0.98853821 1.00277054]
Test:  [-0.01369666  0.84477235  1.81957153 -1.22579757 -0.24522312] [1.0010803  0.90113769 3.06068779 1.54990156 0.58253345]


In [57]:
test_codes = pd.read_parquet('../data/test.parquet')["code"].to_list()
import random 
print(test_codes[random.randint(0, 500000)])

import re
from mitmproxy.coretypes import basethread


def test_basethread():
    t = basethread.BaseThread('foobar')
    assert re.match(r'foobar - age: \d+s', t._threadinfo())


In [ ]:
import pandas as pd
import random

# df = pd.read_parquet('../data/validation.parquet')

human_codes = df[df["label"] == 0]["code"].tolist()
ai_codes    = df[df["label"] == 1]["code"].tolist()

print("=== HUMAN (label = 0) ===")
print(random.choice(human_codes))

print("\n=== AI (label = 1) ===")
print(random.choice(ai_codes))

=== HUMAN (label = 0) ===
for _ in range(int(input())):
	n = int(input())
	a = list(map(int, input().split()))
	for i in range(n - 3):
		target_num = i + 1
		if a[i] == target_num:
			continue
		j = a.index(target_num)
		if j == n - 1:
			a = a[:-3] + a[-1:] + a[-3:-1]
			j = n - 3
		a = a[:i] + a[j:j + 2] + a[i:j] + a[j + 2:]
	tail = a[-3:]
	if tail[0] < tail[1] < tail[2] or tail[1] < tail[2] < tail[0] or tail[2] < tail[0] < tail[1]:
		print('YES')
	else:
		print('NO')


=== AI (label = 1) ===
python
def play_game(s):
    n = len(s)
    reverse_op_cost = 0
    possible_reverse moved = False

    for i in range(n):
        if s[i] == '0':
            s[i] = '1'
        else:
            possible_reverse_moved = True
            break

    if possible_reverse_moved:
        reverse_op_cost = 1

    alice_dollars = 0
    bob_dollars = 0

    turn = 0
    while '0' in s:
        if turn % 2 == 0:
            change_dollars = 1
        else:
            possible_reverse_moved = True
      

### Test blank line feature

In [75]:
def extract_blank_line_feature(code: str):
    if not code or not code.strip():
        return 0.0

    lines = code.strip().split('\n')
    total_lines = len(lines)

    blank_lines = sum(1 for line in lines if not line.strip())
    blank_line_ratio = blank_lines / total_lines if total_lines > 0 else 0.0

    return blank_line_ratio

test_codes = pd.read_parquet('../data/test_sample.parquet')["code"].to_list()
valid_codes = pd.read_parquet('../data/validation.parquet')["code"].to_list()

test_blanks = [extract_blank_line_feature(c) for c in test_codes]
valid_blanks = [extract_blank_line_feature(c) for c in valid_codes]

print(np.mean(test_blanks), np.std(test_blanks))
print(np.mean(valid_blanks), np.std(valid_blanks))

0.08561722476695041 0.1060381270025779
0.1048040715320724 0.12744223712262137


In [ ]:
# blank_ratio > 0.1 is AI
test_preds = []
for blank in test_blanks:
    if (blank > 0.1): test_preds.append(1)
    else: test_preds.append(0)

test_labels = pd.read_parquet('../data/test_sample.parquet')["label"].to_numpy()
correct = np.sum(np.array(test_preds) == test_labels)
print(correct, correct / len(test_labels))

735 0.735


In [78]:
test_codes = pd.read_parquet('../data/test.parquet')["code"].to_list()
blanks = []
preds = []
for c in test_codes:
    blank = extract_blank_line_feature(c)
    blanks.append(blank)
    if blank > 0.1: preds.append(1)
    else: preds.append(0)

ids = pd.read_parquet('../data/test.parquet', columns=["ID"])["ID"].values

df = pd.DataFrame({
    "ID": ids,   # 0 → N-1
    "label": preds
})

df.to_csv("submission.csv", index=False)

In [81]:
print(np.mean(blanks), np.std(blanks))
print(len(preds))

0.08409312881948451 0.10210270725763304
500000
